# 00. 선행연구 기반 대표 빈티지 전환 규칙 검토

이 문서는 K-ETS VWAP 타겟 설계에서 필요한 **대표 빈티지 전환 규칙**의 후보를 정리한다.  
여기서 대표 빈티지 전환 규칙은 금융 시계열에서 말하는 **roll rule** 또는 **롤오버 규칙**을 K-ETS 빈티지 구조에 맞게 적용한 개념이다.
 
데이터 검증은 다음 노트북인 `01_ets_da.ipynb`와 후속 roll rule EDA 노트북에서 수행한다.

## 1. 용어 정의

| 용어 | 본 프로젝트에서의 사용 의미 |
|---|---|
| `roll rule` | 만기 또는 대표 계약을 언제 다음 계약으로 넘길지 정하는 규칙이다. 한국어로는 **롤오버 규칙**, **계약 전환 규칙**, 또는 본 프로젝트 맥락에서는 **대표 빈티지 전환 규칙**이라고 쓴다. |
| `vintage` | 배출권의 이행연도 또는 발행연도 성격을 갖는 구분이다. KRX 데이터에서는 `KAU15`, `KAU16` 같은 `isu_code`로 표현된다. |
| `continuous series` | 여러 만기 또는 여러 빈티지의 가격을 일정한 전환 규칙으로 이어 붙여 만든 단일 시계열이다. |
| `same-vintage panel` | 하나의 대표 시계열을 만들지 않고, 같은 `isu_code` 내부에서만 미래 수익률을 계산하는 패널형 타겟 구성 방식이다. |
| `volume-confirmed roll` | 다음 빈티지의 거래량 우위가 일정 기간 확인될 때만 대표 빈티지를 전환하는 규칙이다. |

K-ETS 데이터에서는 선물 만기 계약이 아니라 KAU 빈티지가 문제의 중심이므로, 본문에서는 `roll rule`을 **대표 빈티지 전환 규칙**으로 표기한다.


## 2. 문헌 사례 1: EU-ETS 연구에서의 현물-선물 가격 관계

EU-ETS는 K-ETS보다 거래 유동성이 높고 선물시장이 발달해 있다. 따라서 EU-ETS 선행연구에서는 EUA 현물 가격과 EUA 선물 가격의 관계가 주요 연구 대상이 된다. Arouri, Jawadi, Nguyen(2012)은 EU-ETS Phase II의 탄소 현물-선물 가격 관계에서 비선형성을 분석하였다. 이 연구는 탄소 가격 시계열을 다룰 때 현물 가격 하나만 볼 것이 아니라, 선물 계약과의 연결 구조를 함께 고려해야 함을 보여준다.

K-ETS의 `KAU15`, `KAU16`, `KAU17` 등은 EU-ETS 선물 계약과 동일한 상품 구조는 아니지만, 예측 타겟을 만들 때 여러 가격 계열 중 어느 계열을 사용할지 정해야 한다는 점에서 유사한 문제를 만든다. 따라서 K-ETS에서도 날짜별 최대 거래량 빈티지를 임의로 선택하는 방식보다, 사전에 정의된 전환 기준을 두는 편이 설명 가능성이 높다.

| 문헌 | 확인 정보 | 본 프로젝트에 주는 시사점 |
|---|---|---|
| Arouri, Jawadi, Nguyen (2012), *Economic Modelling* | `Nonlinearities in carbon spot-futures price relationships during Phase II of the EU ETS`, DOI: `10.1016/j.econmod.2011.11.003` | 탄소 가격은 현물과 선물의 연결 구조를 통해 형성된다. K-ETS에서도 빈티지 간 가격 계열을 연결할 때 전환 기준을 명시해야 한다. |


## 3. 문헌 사례 2: EUA 선물 가격을 직접 예측 대상으로 사용하는 연구

EUA 선물 가격 변동성 예측 연구는 특정 선물 가격 계열을 분석 대상으로 삼는다. Guo et al.(2022)은 EUA futures 변동성을 예측 대상으로 설정했고, Wu, Yin, Mei(2022)는 EUA futures 변동성에 기후정책 불확실성을 결합하였다. 두 사례 모두 예측 대상이 되는 가격 계열이 무엇인지 먼저 정한 뒤, 그 계열의 변동성 또는 수익률을 모델링한다.

K-ETS에서는 동일한 방식으로 거래가 활발한 선물 가격 계열을 바로 사용할 수 없다. 대신 `raw.krx_ets_daily`의 VWAP와 `isu_code`를 이용해 타겟 계열을 구성해야 한다. 이때 대표 빈티지 선택 규칙이 타겟 변수의 정의 일부가 된다. 즉, roll rule은 부가적인 전처리 옵션이 아니라 타겟 정의의 핵심 조건이다.

| 문헌 | 확인 정보 | 본 프로젝트에 주는 시사점 |
|---|---|---|
| Guo et al. (2022), *Energy Economics* | `Forecasting volatility of EUA futures: New evidence`, DOI: `10.1016/j.eneco.2022.106021` | EUA futures 자체가 예측 대상이 될 수 있다. K-ETS에서는 대응되는 단일 가격 계열을 어떻게 만들지 정의해야 한다. |
| Wu, Yin, Mei (2022), *Sustainability* | `Forecasting the Volatility of European Union Allowance Futures with Climate Policy Uncertainty Using the EGARCH-MIDAS Model`, DOI: `10.3390/su14074306` | 일별 시장 가격과 저빈도 정책·불확실성 변수를 결합할 수 있다. K-ETS의 30일/60일 예측 기간 설정도 외생 변수 주기와 함께 설명되어야 한다. |


## 4. 문헌 사례 3: continuous futures price series 방법론

선물시장에서는 여러 만기 계약을 이어 붙여 장기 가격 시계열을 만들 때 continuous futures price series를 사용한다. Went(2010)는 continuous futures price series를 만들 때 계약 전환 시점과 가격 조정 방식이 결과 시계열에 영향을 준다는 점을 정리한다. 핵심은 전환 규칙을 사후적으로 바꾸지 않고, 분석 전에 명시해야 한다는 점이다.

K-ETS의 KAU 빈티지는 선물 만기 계약과 법적 구조가 다르다. 그러나 여러 `isu_code`를 하나의 대표 VWAP 시계열로 이어 붙일 때는 continuous futures 구성과 유사한 문제가 발생한다. 따라서 K-ETS용 대표 빈티지 전환 규칙은 다음 조건을 만족해야 한다.

1. 전환 기준이 사전에 정의되어야 한다.
2. 전환 후 이전 빈티지로 되돌아가는 현상이 없어야 한다.
3. 전환 시점 전후의 가격 점프를 별도로 진단해야 한다.
4. 거래량, 거래대금, 거래 공백 등 시장 유동성 정보를 함께 확인해야 한다.

| 문헌 | 확인 정보 | 본 프로젝트에 주는 시사점 |
|---|---|---|
| Went (2010), SSRN | `Creating Continuous Futures Price Series`, DOI: `10.2139/ssrn.1547310` | 여러 계약을 연결할 때 roll rule이 시계열의 성격을 결정한다. K-ETS 대표 VWAP 시계열도 전환 규칙을 명시해야 한다. |


## 5. 프로젝트 레퍼런스 문서에서 확인한 K-ETS 특성

프로젝트 내부 레퍼런스 문서에서도 K-ETS는 EU-ETS와 다른 구조를 갖는다고 정리되어 있다. `price-models.md`는 EU-ETS의 시장 성숙도와 유동성이 높고 선물 및 파생상품 중심인 반면, K-ETS는 장내 현물 거래 비중이 높고 유동성 부족이 존재한다고 기술한다. 또한 K-ETS의 빈티지는 이월 및 차입 규정과 연결되며, 특정 빈티지의 이행기간 종료나 이월 제한 조치로 인해 빈티지별 특이 현상이 발생할 수 있다고 정리되어 있다.

이 특성은 대표 빈티지 전환 규칙을 설계할 때 중요하다. K-ETS에서 특정 빈티지의 가격 변화는 시장 전체 가격 변화와 제도적 빈티지 효과가 함께 반영될 수 있다. 따라서 30일/60일 VWAP 수익률을 정의할 때 시작 빈티지와 미래 빈티지가 달라지는 경우를 반드시 표시해야 한다.

| 출처 | 확인 내용 | 타겟 설계 반영 |
|---|---|---|
| `docs/reference/price-models.md` | EU-ETS는 선물·파생상품 중심, K-ETS는 장내 현물 비중과 유동성 부족이 존재한다. | EU식 futures series를 그대로 적용하지 않고 K-ETS 빈티지 구조에 맞춘다. |
| `docs/reference/price-models.md` | K-ETS 빈티지는 이월·차입 규정 및 이행기간 종료와 연결된다. | 빈티지 전환 여부를 타겟 보조 컬럼으로 남긴다. |
| `docs/reference/feature-engineering.md` | 탄소 가격 발견은 현물과 선물의 상호작용에 의해 복잡해질 수 있다. | 가격 계열 선택과 외생 변수 결합 주기를 함께 검토한다. |


## 6. K-ETS VWAP 타겟 후보

선행연구 사례와 K-ETS 데이터 구조를 함께 고려하면, 현재 단계에서 비교해야 할 타겟 구성 방식은 다음과 같다.

| 후보 | 정의 | 문헌·시장 관행과의 연결 | 장점 | 주요 한계 | 검증 항목 |
|---|---|---|---|---|---|
| `same_vintage_panel` | 같은 `isu_code` 내부에서만 30일/60일 후 VWAP 로그수익률을 계산한다. | 빈티지 전환 효과를 가격 수익률에서 분리한다. | 해석이 명확하고 빈티지 효과가 섞이지 않는다. | 빈티지별 표본 수가 줄어든다. | 빈티지별 표본 수, 실제 경과일 분포, 극단 수익률 |
| `volume_confirmed_roll` | 다음 빈티지의 거래량 우위가 rolling window와 확인일 조건을 만족하면 대표 빈티지를 전환한다. | continuous futures에서 roll rule을 사전 정의하는 방식과 대응된다. | 유동성 이동을 반영한 단일 대표 시계열을 만들 수 있다. | window와 확인일 수에 민감하다. | rolling 거래량 역전 시점, roll 전후 가격 점프, 되돌림 발생 여부 |
| `calendar_roll` | 연도 또는 이행주기 기준으로 대표 빈티지를 고정 전환한다. | 일부 선물 roll rule의 고정 달력 방식과 유사하다. | 설명과 재현이 쉽다. | 실제 유동성 이동과 다를 수 있다. | 고정 전환일 전후 거래량·VWAP 변화 |
| `daily_max_volume` | 매일 거래량이 가장 큰 빈티지를 대표로 선택한다. | 문헌상 continuous series 구성 원칙과 맞지 않는다. | 진단용으로 계산이 쉽다. | 대표 빈티지가 앞뒤로 바뀔 수 있다. | 되돌림 횟수, 빈티지 변경일 가격 점프 |

현재 1차 기준은 `same_vintage_panel`로 두고, 별도 EDA에서 `volume_confirmed_roll`을 비교 후보로 검토한다. `daily_max_volume`은 최종 타겟 후보가 아니라, 대표 빈티지 전환 문제가 얼마나 큰지 보여주는 진단 지표로 사용한다.


## 7. 30일/60일 예측 기간과 보조 컬럼

월별 또는 저빈도 외생 변수와의 정합성을 고려하면, 주력 타겟 예측 기간은 30일과 60일로 둔다. 이때 수익률은 VWAP 로그수익률로 정의한다.

| 타겟 후보 | 정의 | 해석 |
|---|---|---|
| `target_vwap_logret_30d` | `log(vwap_future_30d / vwap_t)` | 기준일 대비 약 1개월 후 VWAP 변화율 |
| `target_vwap_logret_60d` | `log(vwap_future_60d / vwap_t)` | 기준일 대비 약 2개월 후 VWAP 변화율 |

타겟 데이터에는 다음 보조 컬럼을 함께 둔다.

| 컬럼 | 의미 |
|---|---|
| `future_date_30d`, `future_date_60d` | 목표 기간 이후 실제로 사용된 미래 거래일 |
| `actual_horizon_days_30d`, `actual_horizon_days_60d` | 기준일과 미래 거래일 사이의 실제 경과 일수 |
| `future_isu_code_30d`, `future_isu_code_60d` | 미래 가격에 사용된 빈티지 코드 |
| `is_vintage_changed_30d`, `is_vintage_changed_60d` | 시작 빈티지와 미래 빈티지가 다른지 여부 |
| `roll_rule_version` | 대표 빈티지 전환 규칙의 버전 |

`same_vintage_panel`에서는 `future_isu_code_*`가 시작 `isu_code`와 같아야 한다. `volume_confirmed_roll`에서는 전환 규칙에 따라 달라질 수 있으므로 `is_vintage_changed_*`를 해석에 반드시 사용한다.


## 8. 후속 EDA에서 확인할 항목

후속 노트북에서는 다음 항목을 데이터로 확인한다.

1. 빈티지별 실제 거래 기간과 거래 공백
2. 같은 빈티지 기준 30일/60일 타겟 coverage
3. 같은 빈티지 기준 30일/60일 VWAP 로그수익률 분포와 극단값 날짜
4. 인접 빈티지의 거래 겹침 구간
5. rolling 거래량 기준으로 다음 빈티지가 현재 빈티지를 지속적으로 앞서는 시점
6. 후보 roll rule별 전환일과 전환 전후 VWAP jump
7. 전환 후 이전 빈티지로 되돌아가는 현상이 발생하는지 여부

이 확인을 마친 뒤 `same_vintage_panel`을 기본 타겟으로 둘지, `volume_confirmed_roll` 기반 continuous KAU series를 추가 타겟으로 만들지 결정한다.
